# TD/TP 2 — Image classification with a CNN

**IMT · Introduction to Machine Learning · Google Colab**   
**Based on course material:** Part 2: pixels, channels, convolution, filters, ReLU, pooling, multiclass classification

**Learning objectives:** inspect grayscale images; apply an edge filter; build and train a CNN; analyze image-classification mistakes.

### How to use this notebook
1. Read the questions, run the prepared cells in order, and complete every **✍️ TODO** cell yourself.
2. The data preparation and plotting helpers are supplied so you can focus on machine learning.
3. If you get stuck, discuss your approach with a classmate before consulting the instructor.
4. You will receive the complete solution notebook after the collective correction.

**Environment:** Google Colab → Runtime → Run all

## Scenario: recognizing handwritten digits

We use the **scikit-learn handwritten digits** dataset, included with scikit-learn (**no internet download**). Each image has shape **8 × 8**, grayscale, and label 0–9. These are deliberately small images so that training is fast on a free Colab CPU.

A CNN exploits locality: small filters search for patterns rather than flattening the image immediately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
digits = load_digits()
X = digits.images.astype('float32') / 16.0  # pixel values in [0,1]
y = digits.target.astype('int64')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
xtr = torch.tensor(X_train[:, None, :, :])  # (samples, channels=1, height=8, width=8)
xte = torch.tensor(X_test[:, None, :, :])
ytr = torch.tensor(y_train, dtype=torch.long)
yte = torch.tensor(y_test, dtype=torch.long)
train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(xte, yte), batch_size=128)
fig, axes = plt.subplots(2, 5, figsize=(8,3.5))
for ax, img, label in zip(axes.flat, X_train[:10], y_train[:10]):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set(title=f'Label {label}', xticks=[], yticks=[])
plt.suptitle('Handwritten digits: 1 channel, 8×8 pixels')
plt.tight_layout(); plt.show()
print('Image tensor shape:', tuple(xtr.shape))

## 1. ✍️ TODO — see a convolution in action

A convolution filter scans local neighborhoods. Implement a **vertical edge detector** by filling a 3×3 kernel with values below, then use `torch.nn.functional.conv2d` with `padding=1`.

```
[-1, 0, 1]
[-1, 0, 1]
[-1, 0, 1]
```

**Checkpoint:** output spatial shape remains 8×8. (Deep learning libraries typically implement cross-correlation, often called convolution.)

In [ ]:
import torch.nn.functional as F

# TODO: create `edge_kernel`, a float tensor of shape (1,1,3,3).
raise NotImplementedError('Make the 3x3 filter')

edge_map = F.conv2d(xtr[:1], edge_kernel, padding=1)
assert edge_map.shape == (1, 1, 8, 8)
fig, axs = plt.subplots(1,2,figsize=(6,3))
axs[0].imshow(xtr[0,0], cmap='gray'); axs[0].set_title('Input')
axs[1].imshow(edge_map[0,0], cmap='coolwarm'); axs[1].set_title('Vertical-edge response')
for a in axs: a.axis('off')
plt.show()

## 2. ✍️ TODO — build the CNN architecture

Make a `SmallCNN` with:

1. `Conv2d(1, 8, kernel_size=3, padding=1)` + ReLU + `MaxPool2d(2)` → 8 × 4 × 4
2. `Conv2d(8, 16, kernel_size=3, padding=1)` + ReLU + `MaxPool2d(2)` → 16 × 2 × 2
3. `Flatten` + `Linear(16*2*2, 10)` → **10 logits** for digits 0–9.

The **cross-entropy loss expects logits**, not softmax probabilities.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define `self.net` using the layers described above.
        raise NotImplementedError('Define CNN layers')

    def forward(self, x):
        return self.net(x)

model = SmallCNN()
assert model(xtr[:3]).shape == (3, 10)
print(model)

## 3. ✍️ TODO — train with categorical cross-entropy

Use `nn.CrossEntropyLoss()` and `torch.optim.Adam` with learning rate `0.005`. Fill in the batch-training body. The label tensor contains integer class indices (0–9).

In [ ]:
# TODO: define loss_fn and optimizer.
raise NotImplementedError('Choose the multiclass loss and Adam optimizer')

def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total = 0.
    for xb, yb in loader:
        # TODO: zero_grad -> forward -> loss -> backward -> step
        raise NotImplementedError('Implement the CNN training step')
        total += loss.item()*len(xb)
    return total/len(loader.dataset)

losses = []
for epoch in range(12):
    losses.append(train_epoch(model, train_loader, loss_fn, optimizer))
print('Final training loss:', round(losses[-1], 4))
plt.plot(losses, marker='o'); plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.show()

## 4. ✍️ TODO — evaluate the CNN

Each output row is a vector of 10 logits. Use `argmax(axis=1)` to select the most likely digit and measure **test accuracy**. Then inspect the confusion matrix.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(xte)
    # TODO: turn the logits into `predicted_digits`, a NumPy array.
    raise NotImplementedError('Predict multiclass labels')

assert predicted_digits.shape == (len(X_test),)
accuracy = accuracy_score(y_test, predicted_digits)
print(f'CNN test accuracy: {accuracy:.3f}')
ConfusionMatrixDisplay.from_predictions(y_test, predicted_digits, cmap='Blues')
plt.show()

## 5. ✍️ TODO — inspect first-layer feature maps

Unlike a fully connected network, a CNN learns **filters**. Inspect the output of the very first convolution layer (`model.net[0]`). You should obtain **8 feature maps**, each 8×8.

**Question:** Why do different feature maps respond to different parts of the digit?

In [ ]:
# TODO: compute `maps` for the first test image using the first conv layer.
raise NotImplementedError('Extract CNN feature maps')

assert maps.shape == (1,8,8,8)
fig, axes = plt.subplots(2,4,figsize=(8,4))
for j, ax in enumerate(axes.flat):
    ax.imshow(maps[0,j].detach().numpy(), cmap='viridis')
    ax.set(title=f'Filter {j}',xticks=[],yticks=[])
plt.tight_layout(); plt.show()

## 6. ✍️ TODO — compare parameter counts

Count trainable parameters in the CNN and in a fully connected network `Flatten → Linear(64,128) → ReLU → Linear(128,10)`. You do **not** need to train the MLP for this exercise.

**Question:** Which has fewer parameters, and why? Results on 8×8 digits may not reflect behavior on large images.

In [ ]:
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64,128), nn.ReLU(), nn.Linear(128,10))

def count_parameters(net):
    # TODO: count all elements of parameters with requires_grad=True.
    raise NotImplementedError('Count trainable parameters')

print('CNN trainable parameters:', count_parameters(model))
print('MLP trainable parameters:', count_parameters(mlp))

### Discussion / optional challenge

1. Why does `Conv2d(1, 8, 3)` have **8 output channels**, not 1?
2. Why do we use pooling after convolution?
3. What does the model confuse most often? Look at the confusion matrix.
4. **Challenge:** change the number of filters from 8/16 to 16/32. Do accuracy and parameter count change?

**Takeaway:** CNNs exploit local, shared filters; the full network is trained end-to-end by minimizing cross-entropy.